In [1]:
import os
import time
import datetime
import pandas as pd
import numpy as np
import sys
from ctgan import CTGAN

# ignore all warnings
import warnings
warnings.filterwarnings('ignore')
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("Running on CPU")

PyTorch version: 2.5.1+cu124
CUDA available: True
CUDA version: 12.4
GPU device: NVIDIA RTX 4000 Ada Generation
Number of GPUs: 1


In [2]:
dataset_file  =  "../data/caida-10k.csv"

In [4]:
df = pd.read_csv(dataset_file)

In [5]:
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


Dataset shape: (10000, 15)
Columns: ['srcip', 'dstip', 'srcport', 'dstport', 'proto', 'time', 'pkt_len', 'version', 'ihl', 'tos', 'id', 'flag', 'off', 'ttl', 'chksum']

First few rows:


,srcip,dstip,srcport,dstport,proto,time,pkt_len,version,ihl,tos,id,flag,off,ttl,chksum
0,3992027753,2325249714,443,33195,TCP,1521118750461923,1400,4,5,0,27106,2,0,88,14840
1,3223949995,2332752498,50527,443,TCP,1521118750461924,40,4,5,0,8563,2,0,115,3595
2,1056685515,3618636787,443,27308,TCP,1521118750461924,52,4,5,0,8751,2,0,234,38442
3,284281003,1074109364,80,63403,TCP,1521118750461924,1500,4,5,8,45145,2,0,59,54372
4,568626588,3261365128,443,50749,TCP,1521118750461928,44,4,5,0,0,2,0,54,22367


In [6]:
start_time = time.time()
discrete_columns = ["srcip","dstip","srcport","dstport","proto","version","ihl","tos","flag","off"]
numerical_columns = ["time","pkt_len","id","ttl"]

model_name = "CTGAN"
dataset_name = "caida"


ctgan = CTGAN(epochs=20, verbose=True)
print("starting training")

starting training


In [7]:
ctgan.fit(df, discrete_columns)

Epoch 1, Loss G:  4.0834,Loss D: -0.4629
Epoch 2, Loss G:  4.3149,Loss D: -0.9994
Epoch 3, Loss G:  3.6771,Loss D: -0.7285
Epoch 4, Loss G:  3.1001,Loss D: -0.5120
Epoch 5, Loss G:  3.5889,Loss D: -0.3072
Epoch 6, Loss G:  2.6835,Loss D: -0.0814
Epoch 7, Loss G:  2.1806,Loss D:  0.1596
Epoch 8, Loss G:  2.6772,Loss D:  0.0776
Epoch 9, Loss G:  2.6766,Loss D: -0.0265
Epoch 10, Loss G:  2.5552,Loss D:  0.1078
Epoch 11, Loss G:  2.5783,Loss D:  0.1515
Epoch 12, Loss G:  2.6100,Loss D: -0.0016
Epoch 13, Loss G:  2.2567,Loss D:  0.1493
Epoch 14, Loss G:  2.3773,Loss D: -0.1562
Epoch 15, Loss G:  2.4451,Loss D: -0.0091
Epoch 16, Loss G:  2.5130,Loss D: -0.0077
Epoch 17, Loss G:  2.2065,Loss D: -0.2290
Epoch 18, Loss G:  2.2726,Loss D:  0.0759
Epoch 19, Loss G:  2.4168,Loss D: -0.1398
Epoch 20, Loss G:  2.3863,Loss D: -0.2469


In [8]:
synthetic_data = ctgan.sample(len(df))

In [9]:
print(synthetic_data)

           srcip       dstip  srcport  dstport proto              time  \
0      718026217           0      443    41423   TCP  1521118750476013   
1     1125711061  1235569029       80    50531   TCP  1521118750479660   
2     2503406240  1074109419    10000    62016   TCP  1521118750483237   
3     3307379311  3447595505    29052    38560   UDP  1521118750468889   
4     2270456640  3347704510    32045    15529   TCP  1521118750481449   
...          ...         ...      ...      ...   ...               ...   
9995     3410944  3453656475      443    16454   UDP  1521118750463122   
9996  1559758981  4225200978      443    64610   TCP  1521118750481078   
9997  2211447513           0      443     6881   TCP  1521118750469100   
9998  4234129531  4225200978      443    45442   TCP  1521118750468940   
9999  1962329749  1962334114      443    50848   TCP  1521118750476414   

      pkt_len  version  ihl  tos     id  flag   off  ttl  chksum  
0          -8        4    5    0  -1298     

In [11]:
synthetic_data.to_csv('../data/generated_pcap_ctgan.csv', index=False)